In [0]:
from pyspark.sql import functions as F

## Create bronze schema


Here we create the bronze schema in the final_project catalog.  
All raw NYC taxi tables (train/test) will be stored here as Delta tables.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS final_project.bronze;

## Read raw Kaggle CSVs from Volumes

In this step we read the original Kaggle train.csv and test.csv from the volumes.  
No cleaning or filtering is applied here – we only load the raw data into dataframes.

In [0]:
train_path = f"/Volumes/final_project/default/files/train_2m.csv"
test_path  = f"/Volumes/final_project/default/files/test.csv"

### Persist raw data as bronze Delta tables



We now write the raw DataFrames into Delta tables:

- `final_project.bronze.train`
- `final_project.bronze.test`

These tables represent the source of truth for the rest of the pipeline (silver/gold).

In [0]:
train_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(train_path)
)
(
    train_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("final_project.bronze.train")
)

In [0]:
test_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(test_path)
)
(
    test_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("final_project.bronze.test")
)

## Basic sanity checks on bronze tables

We perform quick checks to confirm that:

- The tables are readable
- The schemas look as expected
- The row counts are non-zero

These checks help validate that ingestion into the bronze layer worked correctly.

In [0]:
%sql
SELECT * FROM final_project.bronze.train LIMIT 10;

In [0]:
train_bronze = spark.table("final_project.bronze.train")

row_count = train_bronze.count()
print(f"Number of rows in final_project.bronze.train: {row_count}")

In [0]:
numeric_types = ("double", "float")
numeric_cols = [c for c, t in train_bronze.dtypes if t in numeric_types]
other_cols   = [c for c, t in train_bronze.dtypes if t not in numeric_types]

null_counts_exprs = [
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in train_bronze.columns
]
null_counts = train_bronze.select(*null_counts_exprs)
print("Null counts per column:")
display(null_counts)

In [0]:
%sql
SELECT * FROM final_project.bronze.test LIMIT 10;

In [0]:
test_bronze = spark.table("final_project.bronze.test")

row_count_test = test_bronze.count()
print(f"Number of rows in final_project.bronze.test: {row_count_test}")